# Colab runner for our Kaggle jobs (free T4)

Runs one of our Kaggle job kernels **unchanged** on a free Colab GPU, for when the weekly Kaggle GPU quota is spent.
It recreates Kaggle's folder layout (`/kaggle/input/...`, `/kaggle/working`), downloads the kernel's code and its data
sources with the Kaggle API, runs it, and uploads `/kaggle/working` as a new version of a private Kaggle dataset.
Claude picks the results up from that dataset.

**One-time setup**
1. Runtime → Change runtime type → **T4 GPU**.
2. Key icon (Secrets) in the left bar → add `KAGGLE_API_TOKEN` with your Kaggle API token, and allow notebook access.

**Each run:** set `KERNEL` in the next cell (Claude tells you which), then Runtime → **Run all**. Keep the tab open.
Free Colab sessions can disconnect after a few idle hours, and outputs only upload at the end, so start long jobs when
you'll be near the laptop.

In [ ]:
KERNEL = "aryanawasthiteykik/shl-headvar"   # the Kaggle job kernel to run (owner/slug)
OUTPUT = KERNEL + "-colab"                   # private dataset the outputs are uploaded to

In [ ]:
import json, os, subprocess, sys
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "kaggle"], check=True)
kaggle = lambda *a: subprocess.run(["kaggle", *a], check=True)
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
# The kernel's code and metadata, then its competition and dataset sources, laid out the way Kaggle mounts them.
kaggle("kernels", "pull", KERNEL, "-p", "/content/job", "-m")
meta = json.load(open("/content/job/kernel-metadata.json"))
for d in ("/kaggle/input", "/kaggle/working"):
    os.makedirs(d, exist_ok=True)
for comp in meta.get("competition_sources", []):
    kaggle("competitions", "download", comp, "-p", f"/kaggle/input/{comp}")
    subprocess.run(f"cd /kaggle/input/{comp} && unzip -q -o '*.zip' && rm -f *.zip", shell=True, check=True)
for ds in meta.get("dataset_sources", []):
    kaggle("datasets", "download", ds, "-p", f"/kaggle/input/{ds.split('/')[1]}", "--unzip")
print("inputs:", os.listdir("/kaggle/input"))

In [ ]:
# Run the job exactly as Kaggle would, from /kaggle/working.
os.chdir("/kaggle/working")
subprocess.run([sys.executable, f"/content/job/{meta['code_file']}"], check=True)

In [ ]:
# Upload /kaggle/working as a new version of the private output dataset (created on the first run).
slug = OUTPUT.split("/")[1][:50]
json.dump({"title": slug, "id": f"{OUTPUT.split('/')[0]}/{slug}", "licenses": [{"name": "CC0-1.0"}]},
          open("/kaggle/working/dataset-metadata.json", "w"))
exists = subprocess.run(["kaggle", "datasets", "status", f"{OUTPUT.split('/')[0]}/{slug}"], capture_output=True).returncode == 0
if exists:
    kaggle("datasets", "version", "-p", "/kaggle/working", "-m", f"colab run of {KERNEL}", "-r", "zip")
else:
    kaggle("datasets", "create", "-p", "/kaggle/working", "-r", "zip")
print("uploaded to", f"https://www.kaggle.com/datasets/{OUTPUT.split('/')[0]}/{slug}")